In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import io
import os
import sys
import textwrap

from collections import Counter

# ---------- 0) Load ----------
path = "df_merged_preprocessed.csv"
df = pd.read_csv(path, low_memory=False)

print("✅ File loaded:", os.path.basename(path))
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print()

# ---------- 1) Basic column checks ----------
print("1) Column name sanity checks")
cols = pd.Series(df.columns, name="original")
trimmed = cols.str.strip()
dupes = trimmed[trimmed.duplicated(keep=False)]
if len(dupes) > 0:
    print(" - ⚠️ Duplicate (after trim) column names detected:")
    print(dupes.to_list())
else:
    print(" - No duplicate column names (after whitespace trim).")

if (cols != trimmed).any():
    changed = pd.DataFrame({"original": cols, "trimmed": trimmed})
    changed = changed[changed["original"] != changed["trimmed"]]
    print(" - ⚠️ Columns with leading/trailing spaces found (original → trimmed):")
    for _, r in changed.iterrows():
        print(f"    • '{r['original']}' → '{r['trimmed']}'")
else:
    print(" - No leading/trailing spaces in column names.")
print()

# ---------- 2) Dtypes & missingness ----------
print("2) Dtypes & missingness overview")
dtypes = df.dtypes.astype(str)
n_missing = df.isna().sum()
missing_pct = (n_missing / len(df))*100
n_unique = df.nunique(dropna=True)

def sample_vals(s, k=5):
    ex = pd.Series(s.dropna().unique()[:k]).astype(str).tolist()
    return ", ".join(ex)

examples = {}
for c in df.columns:
    try:
        examples[c] = sample_vals(df[c], k=5)
    except Exception:
        examples[c] = ""

summary = pd.DataFrame({
    "dtype": dtypes,
    "n_missing": n_missing,
    "missing_pct": missing_pct.round(3),
    "n_unique": n_unique,
    "examples": pd.Series(examples)
}).sort_values(["missing_pct","n_unique"], ascending=[False, True])

# Save & display
summary_path = "/mnt/data/merged_basic_checks_summary.csv"
summary.to_csv(summary_path, index=True, encoding="utf-8-sig")

from caas_jupyter_tools import display_dataframe_to_user
display_dataframe_to_user("Merged file – column summary (dtype, missingness, uniques, examples)", summary)

print(f" - Saved column summary to: {summary_path}")
print()

# ---------- 3) Year column detection & distribution ----------
print("3) Year column detection")
year_candidates = ["년도","연도","year","Year","YEAR","survey_year","SURVEY_YEAR"]
year_col = None
for c in df.columns:
    if c in year_candidates:
        year_col = c
        break

if year_col is None:
    # fuzzy match
    for c in df.columns:
        if "년" in c or "year" in c.lower():
            year_col = c
            break

if year_col:
    print(f" - Detected year column: '{year_col}'")
    yvc = df[year_col].value_counts(dropna=False).sort_index()
    yvc_df = yvc.rename_axis(year_col).reset_index(name="count")
    display_dataframe_to_user("Year distribution", yvc_df)
else:
    print(" - ⚠️ No obvious year column detected.")
print()

# ---------- 4) Duplicate records check ----------
print("4) Duplicate checks")
full_dupes = df.duplicated(keep=False).sum()
print(f" - Full-duplicate rows (all columns identical): {full_dupes:,}")

id_candidates = ["가구일련번호","가구원번호","개인일련번호",
                 "ID","Id","id","respondent_id","RespondentID","RESPONDENT_ID",
                 "개체ID","개체_id"]
subset_keys = []
for c in id_candidates:
    if c in df.columns:
        subset_keys.append(c)
if year_col and year_col not in subset_keys:
    subset_keys.append(year_col)

if subset_keys:
    dupe_subset = df.duplicated(subset=subset_keys, keep=False).sum()
    print(f" - Duplicates on likely key subset {subset_keys}: {dupe_subset:,}")
else:
    print(" - No likely ID columns found to test key-based duplicates.")
print()

# ---------- 5) Mixed-type & numeric-looking in object columns ----------
print("5) Type consistency checks for object columns")
mixed_report = []
for c in df.columns:
    if df[c].dtype == "object":
        s = df[c].dropna().astype(str)
        # numeric-looking ratio
        is_num_like = s.str.fullmatch(r"[+-]?\d+(\.\d+)?").fillna(False)
        num_ratio = is_num_like.mean()
        
        # detect multiple 'shapes' (e.g., list-like, pipe-separated, json-like)
        looks_listy = s.str.contains(r"[\[\],]|;", regex=True).mean()
        looks_pipe = s.str.contains(r"\|").mean()
        
        mixed_report.append({
            "column": c,
            "numeric_like_ratio": round(float(num_ratio), 3),
            "listy_ratio": round(float(looks_listy), 3),
            "pipe_sep_ratio": round(float(looks_pipe), 3),
            "n_unique": int(df[c].nunique(dropna=True)),
            "examples": ", ".join(s.unique()[:3])
        })
mixed_df = pd.DataFrame(mixed_report).sort_values(["numeric_like_ratio","listy_ratio","pipe_sep_ratio"], ascending=False)
display_dataframe_to_user("Object columns – potential type issues (numeric-like, list-like, pipe-separated)", mixed_df)
print()

# ---------- 6) Category drift across years (if year detected) ----------
if year_col:
    print("6) Category drift check (for low-cardinality object columns)")
    # consider categorical if object and <= 30 unique overall
    cat_cols = [c for c in df.columns if df[c].dtype == "object" and df[c].nunique(dropna=True) <= 30]
    drift_rows = []
    years = sorted([v for v in df[year_col].dropna().unique()])
    for c in cat_cols:
        sets = {}
        for y in years:
            vals = set(df.loc[df[year_col]==y, c].dropna().astype(str).unique())
            sets[y] = vals
        # compute min pairwise Jaccard across years
        jaccards = []
        for i in range(len(years)):
            for j in range(i+1, len(years)):
                a, b = sets[years[i]], sets[years[j]]
                if len(a|b) == 0:
                    j = 1.0
                else:
                    j = len(a & b) / len(a | b)
                jaccards.append(j)
        if jaccards:
            min_j = float(np.min(jaccards))
            if min_j < 0.7:  # flag notable drift
                drift_rows.append({
                    "column": c,
                    "min_pairwise_jaccard": round(min_j, 3),
                    "overall_unique": int(df[c].nunique(dropna=True))
                })
    drift_df = pd.DataFrame(drift_rows).sort_values("min_pairwise_jaccard") if drift_rows else pd.DataFrame(columns=["column","min_pairwise_jaccard","overall_unique"])
    display_dataframe_to_user("Potential category drift across years (low Jaccard similarity)", drift_df)
    print(" - Lower Jaccard ⇒ more likely misaligned labels across years.")
    print()
else:
    print("6) Category drift check skipped (no year column).")
    print()

# ---------- 7) Quick numeric sanity (min/max) ----------
print("7) Numeric min/max & outliers (5th/95th percentiles)")
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if num_cols:
    q = df[num_cols].quantile([0.05, 0.95])
    basic_stats = pd.DataFrame({
        "min": df[num_cols].min(),
        "p05": q.loc[0.05],
        "median": df[num_cols].median(),
        "p95": q.loc[0.95],
        "max": df[num_cols].max(),
        "n_missing": df[num_cols].isna().sum()
    })
    display_dataframe_to_user("Numeric columns – min/median/max (+5th/95th percentiles)", basic_stats)
else:
    print(" - No numeric columns detected.")
print()

# ---------- 8) Save a lightweight text report ----------
report_lines = []
report_lines.append(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")
if year_col:
    report_lines.append(f"Year column: {year_col}\n")
    report_lines.append("Year distribution:\n")
    report_lines.append(df[year_col].value_counts(dropna=False).to_string())
    report_lines.append("\n\n")
report_lines.append("Columns (first 30):\n")
report_lines.append(", ".join(df.columns[:30]))
report_lines.append("\n\nMissingness (top 20):\n")
report_lines.append(summary.sort_values("missing_pct", ascending=False).head(20).to_string())

report_text = "\n".join(report_lines)
report_path = "merged_basic_checks_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print(f Saved quick text report: {report_path}")
